### Load the iteration_1_labels_consesus from part 3

In [ ]:
import pandas as pd
from pathlib import Path

file_path = Path("Iteration_1/Step3_Manual_Labeling/iteration_1_labels_consensus.csv")
df = pd.read_csv(file_path)

print(df.shape)
print(df.columns.tolist())
print(df['target_population'].value_counts())

(100, 13)
['username', 'profile_url', 'display_name', 'description', 'location', 'followers_count', 'following_count', 'statuses_count', 'created_at', 'target_population', 'locals_vs_diaspora', 'person_vs_organization', 'consensus_source']
target_population
0    50
2    37
1    13
Name: count, dtype: int64


In [ ]:
print(df['locals_vs_diaspora'].value_counts())
print()
print(df['person_vs_organization'].value_counts())

locals_vs_diaspora
2    92
1     6
0     2
Name: count, dtype: int64

person_vs_organization
1    48
2    33
0    19
Name: count, dtype: int64


### Translation Cell 

In [ ]:
from deep_translator import GoogleTranslator
from pathlib import Path
import time

# 1) Decide where to save the translated file so we only translate ONCE.
out_dir = Path('Iteration_1/Step5_Analysis')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'iteration_1_consensus_translated.csv'

# 2) If the cache file already exists, skip translation and just load it.
if out_path.exists():
    df = pd.read_csv(out_path)
    print(f"Loaded {len(df)} users from translation cache. Skipping translation.")

else:
    # 3) Helper that translates one piece of text safely.
    def translate_to_english(text):
        if pd.isna(text) or str(text).strip() == '':
            return ''
        try:
            return GoogleTranslator(source='auto', target='en').translate(str(text)[:4500])
        except Exception:
            return str(text)

    # 4) Translate every description.
    print("Translating descriptions...")
    df['description_en'] = ''
    for i, val in enumerate(df['description']):
        df.at[i, 'description_en'] = translate_to_english(val)
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(df)} done")
        time.sleep(0.1)

    # 5) Translate every display_name.
    print("\nTranslating display names...")
    df['display_name_en'] = ''
    for i, val in enumerate(df['display_name']):
        df.at[i, 'display_name_en'] = translate_to_english(val)
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(df)} done")
        time.sleep(0.1)

    # 6) Save the result so we never have to redo this slow step.
    df.to_csv(out_path, index=False)
    print(f"\nSaved translated data to: {out_path}")

# 7) Quick sanity check — show 5 rows with original + translated text side by side.
df[['username', 'description', 'description_en', 'display_name', 'display_name_en']].head(5)

Loaded 100 users from translation cache. Skipping translation.


,username,description,description_en,display_name,display_name_en
0,a_s_v_t_r,اللَّهُمَّ لَا تَدَع لي أمْراً إِلَّا يَسَّرَت...,"Oh God, do not leave for me a matter without m...",حــزامـ,Belt
1,ali_tavakoli_28,سر سری رد شو و زندگی کن.دقت دق ات میدهد.,Go ahead and live. Carefulness pays attention ...,hopkinz,hopkinz
2,adryn848,NaN,NaN,آدرین,Adrienne
3,alexmarlow,**BREAKING THE LAW** OUT NOW! 'The Alex Marlow...,**BREAKING THE LAW** OUT NOW! 'The Alex Marlow...,Alex Marlow,Alex Marlow
4,aaihs,AAIHS was founded to foster dialogue about #Bl...,AAIHS was founded to foster dialogue about #Bl...,AAIHS,AAIHS


In [ ]:
import numpy as np

# Make sure the count columns are numeric (they might be read as strings if there were any non-numeric values)

for col in ['followers_count', 'following_count', 'statuses_count']:
    # Convert to numeric, set non-convertible values to NaN
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Bio Length - using the translated English description for consistency
df['bio_length'] = df['description_en'].fillna('').str.len()

# Followers-to-Following Ratio - add 1 to denominator to avoid division by zero
df['followers_following_ratio'] = df['followers_count'] / (df['following_count'] + 1)

# Account age in years - using 'created_at' column
df['created_at_dt'] = pd.to_datetime(df['created_at'], format='%B %Y', errors='coerce')
df['account_age_years'] = ((pd.Timestamp.today() - df['created_at_dt']).dt.days/ 365.25).fillna(0)

# Binary flags, 1 if description or location is present, else 0
df['has_description'] = df['description'].notna().astype(int)
df['has_location']    = df['location'].notna().astype(int)

# Iran keyword flags in description and location (case-insensitive)
iran_keywords = ['iran', 'iranian', 'persian', 'persia', 'tehran', 'shiraz', 'esfahan',
                   'isfahan', 'mashhad', 'tabriz', 'kerman', 'qom', 'farsi']

def mentions_iran(text):
    if pd.isna(text):
        return 0
    text_lower = text.lower()
    return int(any(keyword in text_lower for keyword in iran_keywords))


df['bio_mentions_iran']      = df['description_en'].apply(mentions_iran)
df['name_mentions_iran']     = df['display_name_en'].apply(mentions_iran)
df['location_mentions_iran'] = df['location'].apply(mentions_iran)


# Collect the names of your 11 numerical features into a list for easy reference later.
numerical_features = [
    'followers_count',
    'following_count',
    'statuses_count',
    'bio_length',
    'followers_following_ratio',
    'account_age_years',
    'has_description',
    'has_location',
    'bio_mentions_iran',
    'name_mentions_iran',
    'location_mentions_iran'
]

print(f"Built {len(numerical_features)} numeric features")
print(df[numerical_features].describe().round(2))




Built 11 numeric features
       followers_count  following_count  statuses_count  bio_length  \
count           100.00           100.00          100.00      100.00   
mean           1006.31           921.75         1383.85       65.58   
std            2014.59          1456.92         2038.59       61.87   
min               0.00             0.00            0.00        0.00   
25%               0.00            61.50            7.00        0.00   
50%              26.50           349.50          411.50       50.00   
75%             543.25           812.00         1401.50      132.25   
max            8556.00          7496.00         8467.00      182.00   

       followers_following_ratio  account_age_years  has_description  \
count                     100.00             100.00           100.00   
mean                       55.95               7.55             0.73   
std                       350.02               5.81             0.45   
min                         0.00              

Cell 4: build 7 TF-IDF feature sets

  This is where you turn the text (description + username + display_name) into
  numbers a model can read. 

  1. description alone 
  2. username alone
  3. display_name alone
  4. description + username
  5. description + display_name
  6. username + display_name
  7. All three concatenated

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer


def build_tfidf(columns, min_df=2):
    """Combine the given df columns into one text per row, then run TF-IDF on it."""
    text = df[columns[0]].fillna('').astype(str)
    for c in columns[1:]:
        text = text + ' ' + df[c].fillna('').astype(str)
    vec = TfidfVectorizer(max_features=300, lowercase=True, stop_words='english', min_df=min_df)
    matrix = vec.fit_transform(text)
    return matrix, vec


# 7 TF-IDF feature sets per PDF page 12.
# min_df=2 where description is involved (shared vocabulary across users).
# min_df=1 for username/fullname only (mostly unique per user).
tfidf_sets = {
    'desc': build_tfidf(['description_en'], min_df=2),
    'username': build_tfidf(['username'], min_df=1),
    'fullname': build_tfidf(['display_name_en'], min_df=1),
    'desc_user': build_tfidf(['description_en', 'username'], min_df=2),
    'desc_fullname': build_tfidf(['description_en', 'display_name_en'], min_df=2),
    'user_fullname': build_tfidf(['username', 'display_name_en'], min_df=1),
    'desc_user_fullname': build_tfidf(['description_en', 'username', 'display_name_en'], min_df=2),
}


print("TF-IDF feature sets:")
for name, (matrix, vec) in tfidf_sets.items():
    print(f"  {name:<22s}  shape={matrix.shape}  vocab_size={len(vec.vocabulary_)}")

TF-IDF feature sets:
  desc                    shape=(100, 51)  vocab_size=51
  username                shape=(100, 100)  vocab_size=100
  fullname                shape=(100, 173)  vocab_size=173
  desc_user               shape=(100, 51)  vocab_size=51
  desc_fullname           shape=(100, 64)  vocab_size=64
  user_fullname           shape=(100, 268)  vocab_size=268
  desc_user_fullname      shape=(100, 64)  vocab_size=64


In [ ]:
import scipy.sparse as sp
from sklearn.preprocessing import StandardScaler

# 1) Scale the 11 numeric features so they're on similar magnitudes.
#    with_mean=False keeps the matrix sparse-friendly (won't shift values negative).
scaler = StandardScaler(with_mean=False)
numeric_scaled = sp.csr_matrix(scaler.fit_transform(df[numerical_features].values))

# 2) Build the 9 feature sets:
#    - 7 TF-IDF sets from Cell 4
#    - 1 numeric-only set
#    - 1 combined: description TF-IDF stacked with numeric (often the strongest)
feature_sets = {name: matrix for name, (matrix, vec) in tfidf_sets.items()}
feature_sets['numeric'] = numeric_scaled
feature_sets['desc+numeric'] = sp.hstack([tfidf_sets['desc'][0], numeric_scaled]).tocsr()

# 3) Sanity check
print(f"Total feature sets: {len(feature_sets)}")
for name, matrix in feature_sets.items():
    print(f"  {name:<22s} shape={matrix.shape}")

Total feature sets: 9
  desc                   shape=(100, 51)
  username               shape=(100, 100)
  fullname               shape=(100, 173)
  desc_user              shape=(100, 51)
  desc_fullname          shape=(100, 64)
  user_fullname          shape=(100, 268)
  desc_user_fullname     shape=(100, 64)
  numeric                shape=(100, 11)
  desc+numeric           shape=(100, 62)


### Cell 6: The experiment helper function

`run_one_experiment(...)` does one full experiment (train + cross-validate + score) and returns one dictionary that becomes one row of the results CSV. The main loop in Cell 7 will call this function for every combination of (task, classes, algorithm, feature set, CV, balance).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, LeaveOneOut
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# XGBoost needs OpenMP on macOS. If the import fails, skip it gracefully.
try:
    from xgboost import XGBClassifier
    XGBOOST_OK = True
except Exception as e:
    XGBOOST_OK = False
    print(f"XGBoost not available ({type(e).__name__}). Run `brew install libomp` if you want it.")


def make_model(algo_name, balanced):
    """Create a fresh classifier. `balanced` switches on class_weight for algos that support it."""
    cw = 'balanced' if balanced else None
    if algo_name == 'LogReg':
        return LogisticRegression(max_iter=2000, class_weight=cw, random_state=42)
    if algo_name == 'DecisionTree':
        return DecisionTreeClassifier(class_weight=cw, random_state=42)
    if algo_name == 'RandomForest':
        return RandomForestClassifier(n_estimators=100, class_weight=cw, random_state=42, n_jobs=-1)
    if algo_name == 'SVM':
        # Linear kernel is fast on sparse TF-IDF and supports probability output for AUC.
        return SVC(kernel='linear', probability=True, class_weight=cw, random_state=42)
    if algo_name == 'AdaBoost':
        # AdaBoost has no class_weight — we'll pass sample_weight at fit time if balanced.
        return AdaBoostClassifier(random_state=42)
    if algo_name == 'XGBoost':
        return XGBClassifier(eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1)
    raise ValueError(f"Unknown algorithm: {algo_name}")


def class_counts(y):
    """Return {0: n0, 1: n1, 2: n2}, filling missing classes with 0."""
    c = pd.Series(y).value_counts().to_dict()
    return {0: int(c.get(0, 0)), 1: int(c.get(1, 0)), 2: int(c.get(2, 0))}


def sample_weights_for_balance(y):
    """Per-row weights so each class contributes equally during fit (for AdaBoost / XGBoost)."""
    y = np.asarray(y)
    classes, counts = np.unique(y, return_counts=True)
    w = {c: len(y) / (len(classes) * cnt) for c, cnt in zip(classes, counts)}
    return np.array([w[v] for v in y])


def run_one_experiment(X, y, algo_name, balanced, training_type, target_column, feature_set_name, n_classes):
    """Run ONE experiment and return a dictionary that becomes one row in the results CSV."""
    y = np.asarray(y)

    # Choose CV strategy
    if training_type == 'K-Fold':
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        K_val = 5
    else:
        cv = LeaveOneOut()
        K_val = ''

    all_true, all_pred = [], []
    proba_chunks = []

    accuracy = precision = recall = f1 = auc = None
    try:
        for train_idx, test_idx in cv.split(np.zeros(len(y)), y):
            X_tr, X_te = X[train_idx], X[test_idx]
            y_tr, y_te = y[train_idx], y[test_idx]
            if len(np.unique(y_tr)) < 2:
                continue  # degenerate fold

            model = make_model(algo_name, balanced)
            fit_kwargs = {}
            if balanced and algo_name in ('AdaBoost', 'XGBoost'):
                fit_kwargs['sample_weight'] = sample_weights_for_balance(y_tr)
            model.fit(X_tr, y_tr, **fit_kwargs)

            y_pred = model.predict(X_te)
            all_pred.extend(y_pred)
            all_true.extend(y_te)

            if hasattr(model, 'predict_proba'):
                proba_chunks.append((model.classes_, model.predict_proba(X_te), list(test_idx)))

        if not all_pred:
            raise RuntimeError("no predictions made")

        accuracy  = accuracy_score(all_true, all_pred)
        precision = precision_score(all_true, all_pred, average='weighted', zero_division=0)
        recall    = recall_score(all_true, all_pred, average='weighted', zero_division=0)
        f1        = f1_score(all_true, all_pred, average='weighted', zero_division=0)

        # AUC: reassemble the per-fold probabilities into a single (n_samples, n_classes) matrix
        if proba_chunks:
            global_classes = sorted(set(y))
            proba_full = np.zeros((len(y), len(global_classes)))
            for classes_seen, block, idxs in proba_chunks:
                col_map = [global_classes.index(c) for c in classes_seen]
                for k, idx in enumerate(idxs):
                    for j, col in enumerate(col_map):
                        proba_full[idx, col] = block[k, j]
            try:
                if len(global_classes) == 2:
                    auc = roc_auc_score(y, proba_full[:, 1])
                else:
                    auc = roc_auc_score(y, proba_full, multi_class='ovr', average='weighted')
            except Exception:
                auc = None
    except Exception:
        pass

    counts = class_counts(y)
    if n_classes == 2:
        nonzero = [v for k, v in counts.items() if k != 2 and v > 0]
    else:
        nonzero = [v for v in counts.values() if v > 0]
    min_size = min(nonzero) if nonzero else 0

    return {
        'iteration':       1,
        'target_column':   target_column,
        '#classes':        n_classes,
        '#class_0':        counts[0],
        '#class_1':        counts[1],
        '#class_2':        counts[2] if n_classes == 3 else 0,
        'min_class_size':  min_size,
        'training_type':   training_type,
        'K':               K_val,
        'algorithm':       algo_name,
        'feature_set':     feature_set_name,
        'Features_count':  X.shape[1],
        'balanced':        balanced,
        'accuracy':        accuracy,
        'precision':       precision,
        'recall':          recall,
        'F1':              f1,
        'AUC':             auc,
    }


# Quick test on one combination
ALGOS = ['LogReg', 'DecisionTree', 'RandomForest', 'SVM', 'AdaBoost']
if XGBOOST_OK:
    ALGOS.append('XGBoost')

print(f"Algorithms ready: {ALGOS}\n")
print("Smoke test on one experiment (target_population, 3-class, LogReg, numeric, K-Fold, balanced):")
test_row = run_one_experiment(
    X=feature_sets['numeric'],
    y=df['target_population'].values,
    algo_name='LogReg',
    balanced=True,
    training_type='K-Fold',
    target_column='target_population',
    feature_set_name='numeric',
    n_classes=3,
)
for k, v in test_row.items():
    print(f"  {k:<18s} {v}")

Algorithms ready: ['LogReg', 'DecisionTree', 'RandomForest', 'SVM', 'AdaBoost', 'XGBoost']

Smoke test on one experiment (target_population, 3-class, LogReg, numeric, K-Fold, balanced):
  iteration          1
  target_column      target_population
  #classes           3
  #class_0           50
  #class_1           13
  #class_2           37
  min_class_size     13
  training_type      K-Fold
  K                  5
  algorithm          LogReg
  feature_set        numeric
  Features_count     11
  balanced           True
  accuracy           0.64
  precision          0.6689767441860465
  recall             0.64
  F1                 0.6493743890518084
  AUC                0.7502180623973729


### Cell 7: The experiment loop 

3 tasks × 2 class-counts × 6 algorithms × 9 feature sets × 2 CV strategies × 2 balance modes = **1,296 experiments**. Saves the CSV every 50 rows so progress isn't lost. The full CSV is the Step 5 deliverable (PDF page 13).

In [ ]:
import time

# The three classification tasks (PDF page 10)
TASKS = [
    ('target_population',      df['target_population'].values),
    ('locals_vs_diaspora',     df['locals_vs_diaspora'].values),
    ('person_vs_organization', df['person_vs_organization'].values),
]

# Exact column order the PDF page 13 demands
COL_ORDER = [
    'iteration', 'target_column', '#classes', '#class_0', '#class_1', '#class_2',
    'min_class_size', 'training_type', 'K', 'algorithm', 'feature_set',
    'Features_count', 'balanced', 'accuracy', 'precision', 'recall', 'F1', 'AUC',
]

CSV_PATH = Path('experiments_results_iteration_1.csv')

results = []
t0 = time.time()


def save_progress():
    pd.DataFrame(results)[COL_ORDER].to_csv(CSV_PATH, index=False)


# The 6 nested loops
for target_name, y_full in TASKS:
    for n_classes in [3, 2]:
        # 2-class mode: drop the 'unknown' rows (label == 2)
        if n_classes == 2:
            mask = (y_full != 2)
            y = y_full[mask]
        else:
            mask = None
            y = y_full

        if len(np.unique(y)) < 2:
            print(f"  Skipping {target_name} ({n_classes}cls): only one class present")
            continue

        for fset_name, X_full in feature_sets.items():
            X = X_full[mask] if mask is not None else X_full
            for algo_name in ALGOS:
                for training_type in ['K-Fold', 'LOOCV']:
                    for balanced in [True, False]:
                        row = run_one_experiment(
                            X, y, algo_name, balanced, training_type,
                            target_name, fset_name, n_classes,
                        )
                        results.append(row)
                        # Save every 50 experiments so we never lose more than 50 rows of work
                        if len(results) % 50 == 0:
                            save_progress()
                            elapsed = time.time() - t0
                            print(f"  ... {len(results)} done  ({elapsed:.0f}s elapsed)  [partial CSV saved]")

# Final save (catches the last <50 rows)
save_progress()
print(f"\nDone. Total experiments: {len(results)} in {time.time()-t0:.0f}s")
print(f"Saved to: {CSV_PATH}")

KeyboardInterrupt: 

### Cell 8: Read the results and pick winners

Reload the CSV, do a sanity check, show top performers per task, and call out degenerate models (high accuracy + low AUC) so you can talk honestly about them in the report.

In [ ]:
# 1) Reload the CSV from disk
results = pd.read_csv('experiments_results_iteration_1.csv')

print(f"=== SANITY CHECK ===")
print(f"Total rows: {len(results)}")
print(f"Columns: {results.columns.tolist()}")
print(f"NaN counts in metric columns:")
print(results[['accuracy', 'precision', 'recall', 'F1', 'AUC']].isna().sum())
print()

# 2) Top 5 by F1 per task (3-class — the realistic version with 'unknown' kept)
print("=== TOP 5 BY F1 PER TASK (3-class, all rows) ===")
for tgt in ['target_population', 'locals_vs_diaspora', 'person_vs_organization']:
    print(f"\n>>> {tgt}")
    sub = (results[(results['target_column'] == tgt) & (results['#classes'] == 3)]
           .sort_values('F1', ascending=False))
    print(sub[['algorithm', 'feature_set', 'training_type', 'balanced',
                'accuracy', 'F1', 'AUC']].head(5).round(3).to_string(index=False))

# 3) Honest winners — same ranking but with AUC > 0.6 filter to drop degenerate models
print("\n\n=== HONEST WINNER PER TASK (3-class, AUC > 0.6 filter) ===")
winners_rows = []
for tgt in ['target_population', 'locals_vs_diaspora', 'person_vs_organization']:
    sub = (results[(results['target_column'] == tgt) & (results['#classes'] == 3) & (results['AUC'] > 0.6)]
           .sort_values(['F1', 'AUC'], ascending=False))
    if sub.empty:
        print(f"\n>>> {tgt}: NO non-degenerate model found (AUC > 0.6). This task is unlearnable at n=100.")
        winners_rows.append({
            'task': tgt, 'algorithm': '(no honest model)', 'feature_set': '-',
            'training_type': '-', 'balanced': '-',
            'accuracy': None, 'F1': None, 'AUC': None,
            'note': 'No model with AUC > 0.6 — degenerate at n=100',
        })
    else:
        row = sub.iloc[0]
        print(f"\n>>> {tgt}")
        print(f"  Algorithm:      {row['algorithm']}")
        print(f"  Feature set:    {row['feature_set']}")
        print(f"  Training type:  {row['training_type']}")
        print(f"  Balanced:       {row['balanced']}")
        print(f"  Accuracy:       {row['accuracy']:.3f}")
        print(f"  F1:             {row['F1']:.3f}")
        print(f"  AUC:            {row['AUC']:.3f}")
        winners_rows.append({
            'task': tgt,
            'algorithm': row['algorithm'],
            'feature_set': row['feature_set'],
            'training_type': row['training_type'],
            'balanced': row['balanced'],
            'accuracy': round(row['accuracy'], 4),
            'F1': round(row['F1'], 4),
            'AUC': round(row['AUC'], 4),
            'note': 'Honest winner (AUC > 0.6)',
        })

# 4) Save the summary CSV
winners_df = pd.DataFrame(winners_rows)
winners_path = Path('iteration_1_winners_summary.csv')
winners_df.to_csv(winners_path, index=False)
print(f"\n\nSaved winners summary to: {winners_path}")
print(winners_df.to_string(index=False))

# 5) Spot degenerate models (high accuracy but low AUC = predicting majority class)
print("\n\n=== DEGENERATE MODELS (accuracy > 0.85 BUT AUC < 0.55) ===")
degen = results[(results['accuracy'] > 0.85) & (results['AUC'] < 0.55)]
print(f"Found {len(degen)} degenerate rows out of {len(results)} total.")
print("These models look great by accuracy but are just predicting the majority class.")
print(f"Distribution by task:")
print(degen['target_column'].value_counts())

=== SANITY CHECK ===
Total rows: 1296
Columns: ['iteration', 'target_column', '#classes', '#class_0', '#class_1', '#class_2', 'min_class_size', 'training_type', 'K', 'algorithm', 'feature_set', 'Features_count', 'balanced', 'accuracy', 'precision', 'recall', 'F1', 'AUC']
NaN counts in metric columns:
accuracy     0
precision    0
recall       0
F1           0
AUC          0
dtype: int64

=== TOP 5 BY F1 PER TASK (3-class, all rows) ===

>>> target_population
algorithm  feature_set training_type  balanced  accuracy    F1   AUC
   LogReg      numeric        K-Fold     False      0.67 0.665 0.747
 AdaBoost      numeric        K-Fold      True      0.65 0.654 0.723
   LogReg      numeric         LOOCV     False      0.66 0.651 0.735
      SVM      numeric        K-Fold      True      0.64 0.651 0.781
   LogReg desc+numeric         LOOCV     False      0.66 0.650 0.740

>>> locals_vs_diaspora
algorithm  feature_set training_type  balanced  accuracy    F1   AUC
 AdaBoost desc+numeric        